# 🏨 Export Perhitungan SAW — Data Real

Notebook ini memuat **data real** dari folder `data/analyzed/` dan mengekspor setiap tahap perhitungan SAW ke `.csv`.

### Bobot Kriteria
| Kode | Kriteria | Jenis | Bobot | Maks Weighted |
|------|----------|-------|-------|---------------|
| C1 | Fasilitas Hotel | Benefit | 0.45 | ≤ 0.45 |
| C2 | Score Sentiment | Benefit | 0.30 | ≤ 0.30 |
| C3 | Rating | Benefit | 0.25 | ≤ 0.25 |
| | | **Total** | **1.00** | ≤ **1.00** |

In [ ]:
import pandas as pd
import os
import json
import glob

os.makedirs('output_csv', exist_ok=True)

## Fungsi Bantu

In [ ]:
def calculate_c1_score(favorite_features):
    """Hitung skor C1 dari kategori fasilitas. Nilai mentah maks = 0.45"""
    if not favorite_features: return 0.0
    features = [f.lower() for f in favorite_features]
    score = 0.0
    if any('wifi' in f or 'wi-fi' in f or 'internet' in f for f in features):                                                          score += 0.10
    if any(kw in f for f in features for kw in ['breakfast', 'coffee shop', 'halal', 'restaurant', 'bar', 'kitchen']):                 score += 0.15
    if any(kw in f for f in features for kw in ['room service', 'housekeeping', 'air conditioning', 'balcony', 'terrace']):            score += 0.10
    if any(kw in f for f in features for kw in ['airport', 'shuttle', 'car park']):                                                    score += 0.05
    if any(kw in f for f in features for kw in ['fitness', 'pool', 'sauna', 'massage', 'garden', 'beach', 'tours']):                  score += 0.03
    if any(kw in f for f in features for kw in ['elevator', 'check-in', 'lounge', 'safety', 'smoking', 'concierge']):                 score += 0.02
    return score

def calculate_average_rating(reviews):
    """Rata-rata rating dari ulasan. Skala mentah 0–10."""
    if not reviews: return 0.0
    total = sum(r['rating'] for r in reviews if r.get('rating') is not None)
    count = sum(1 for r in reviews if r.get('rating') is not None)
    return total / count if count > 0 else 0.0

def get_avg_sentiment(hotel):
    """Ambil rata-rata skor sentimen. Skala mentah 0–100."""
    if hotel.get('averageSentimentScore') is not None:
        return hotel['averageSentimentScore']
    reviews = hotel.get('reviews', [])
    valid = [r['sentimentScore'] for r in reviews if r.get('sentimentScore') is not None]
    return sum(valid) / len(valid) if valid else 0.0

print('Fungsi bantu berhasil didefinisikan.')

## Tahap 1: Matriks Keputusan (Nilai Mentah)
Memuat seluruh data hotel dan menghitung nilai mentah C1, C2, C3.

> **Perhatian**: Nilai C2 (0–100) dan C3 (0–10) masih dalam skala aslinya. Ini **wajar** dan **belum dibandingkan dengan bobot**.

In [ ]:
hotel_files = glob.glob('data/analyzed/*.json')
hotels_data = []

for f in hotel_files:
    with open(f, 'r', encoding='utf-8') as file:
        data = json.load(file)
    hotels_data.append({
        'name'  : data.get('name', 'Unknown'),
        'c1_raw': calculate_c1_score(data.get('favoriteFeatures', [])),  # Skala 0 - 0.45
        'c2_raw': get_avg_sentiment(data),                                # Skala 0 - 100
        'c3_raw': calculate_average_rating(data.get('reviews', []))       # Skala 0 - 10
    })

df_raw = pd.DataFrame(hotels_data)

print('==== TAHAP 1: MATRIKS KEPUTUSAN (NILAI MENTAH) ====')
print(f'Total hotel: {len(df_raw)}')
print(f'  max C1 (Fasilitas) = {df_raw["c1_raw"].max():.4f}  (skala asli: 0 - 0.45)')
print(f'  max C2 (Sentimen)  = {df_raw["c2_raw"].max():.4f}  (skala asli: 0 - 100)')
print(f'  max C3 (Rating)    = {df_raw["c3_raw"].max():.4f}  (skala asli: 0 - 10)')
print()
display(df_raw.head(10))

df_raw.to_csv('output_csv/3_1_real_saw_raw_matrix.csv', index=False)
print('\nDiekspor ke: output_csv/3_1_real_saw_raw_matrix.csv')

## Tahap 2: Normalisasi Matriks (Skala 0–1)

$$R_{ij} = \frac{X_{ij}}{\max_j(X_{ij})}$$

Setelah tahap ini, **semua nilai C1, C2, C3 pasti berada di antara 0.0 dan 1.0**.

In [ ]:
max_c1 = max(df_raw['c1_raw'].max(), 0.0001)
max_c2 = max(df_raw['c2_raw'].max(), 0.0001)
max_c3 = max(df_raw['c3_raw'].max(), 0.0001)

df_norm = df_raw.copy()
df_norm['c1_norm'] = df_norm['c1_raw'] / max_c1   # Hasil: 0.0 - 1.0
df_norm['c2_norm'] = df_norm['c2_raw'] / max_c2   # Hasil: 0.0 - 1.0
df_norm['c3_norm'] = df_norm['c3_raw'] / max_c3   # Hasil: 0.0 - 1.0

df_norm_view = df_norm[['name', 'c1_norm', 'c2_norm', 'c3_norm']]

print('==== TAHAP 2: MATRIKS NORMALISASI (Skala 0.0 - 1.0) ====')
print(f'  max c1_norm = {df_norm["c1_norm"].max():.4f} | max c2_norm = {df_norm["c2_norm"].max():.4f} | max c3_norm = {df_norm["c3_norm"].max():.4f}')
print()
display(df_norm_view.head(10))

df_norm_view.to_csv('output_csv/3_2_real_saw_normalized.csv', index=False)
print('\nDiekspor ke: output_csv/3_2_real_saw_normalized.csv')

## Tahap 3: Perkalian Bobot (Nilai Preferensi per Kriteria)

$$V_{Cki} = W_{Ck} \times R_{Cki}$$

Batas atas yang **dijamin tidak terlampaui**:
- `c1_weighted` ≤ **0.45**  
- `c2_weighted` ≤ **0.30**  
- `c3_weighted` ≤ **0.25**

In [ ]:
W_C1, W_C2, W_C3 = 0.45, 0.30, 0.25

df_weighted = df_norm.copy()
df_weighted['c1_weighted'] = W_C1 * df_weighted['c1_norm']   # Maks: 0.45
df_weighted['c2_weighted'] = W_C2 * df_weighted['c2_norm']   # Maks: 0.30
df_weighted['c3_weighted'] = W_C3 * df_weighted['c3_norm']   # Maks: 0.25

df_weighted_view = df_weighted[['name', 'c1_weighted', 'c2_weighted', 'c3_weighted']]

c1_ok = df_weighted['c1_weighted'].max() <= 0.45
c2_ok = df_weighted['c2_weighted'].max() <= 0.30
c3_ok = df_weighted['c3_weighted'].max() <= 0.25

print('==== TAHAP 3: NILAI PREFERENSI PER KRITERIA (Setelah Dikali Bobot) ====')
print(f'  c1_weighted maks = {df_weighted["c1_weighted"].max():.6f}  <= 0.45? {"YA" if c1_ok else "TIDAK - ADA BUG!"}')
print(f'  c2_weighted maks = {df_weighted["c2_weighted"].max():.6f}  <= 0.30? {"YA" if c2_ok else "TIDAK - ADA BUG!"}')
print(f'  c3_weighted maks = {df_weighted["c3_weighted"].max():.6f}  <= 0.25? {"YA" if c3_ok else "TIDAK - ADA BUG!"}')
print()
display(df_weighted_view.head(10))

df_weighted_view.to_csv('output_csv/3_3_real_saw_weighted_per_criteria.csv', index=False)
print('\nDiekspor ke: output_csv/3_3_real_saw_weighted_per_criteria.csv')

## Tahap 4: V-Score Akhir & Ranking

$$V_i = c1\_weighted_i + c2\_weighted_i + c3\_weighted_i \quad (\text{maks} = 1.0)$$

In [ ]:
df_final = df_weighted.copy()
df_final['saw_score'] = df_final['c1_weighted'] + df_final['c2_weighted'] + df_final['c3_weighted']
df_final['rank'] = df_final['saw_score'].rank(ascending=False).astype(int)

df_final_view = df_final.sort_values('rank')[['rank', 'name', 'saw_score', 'c1_weighted', 'c2_weighted', 'c3_weighted']]

print('==== TAHAP 4: V-SCORE AKHIR & RANKING ====')
print(f'  saw_score tertinggi = {df_final["saw_score"].max():.6f}  (maks teoritis: 1.0)')
print(f'  saw_score terendah  = {df_final["saw_score"].min():.6f}')
print()
display(df_final_view.head(20))

df_final_view.to_csv('output_csv/3_4_real_saw_final_ranking.csv', index=False)
print('\nDiekspor ke: output_csv/3_4_real_saw_final_ranking.csv')